# 03 - Novel ('brain-storm') factor research

Eight non-textbook factors live in `mfa.factors.novel`. For each we restate the
**behavioural story**, then test whether it is actually priced in our universe.

If a factor's IC has the OPPOSITE sign of the story but is still significant, that's a
real signal -- we just flip its sign in the combiner.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from mfa.config import load_config
from mfa.viz import apply_style
apply_style()

cfg = load_config(ROOT / "configs" / "default.yaml")
print("project:", cfg.project_root.name, "| processed:", cfg.processed_dir.exists())


In [ ]:
from mfa.data import build_panels

def _load_raw_panels():
    out = {}
    for k in ["open","high","low","close","volume","vwap","dollar_volume"]:
        p = cfg.processed_dir / f"panel_{k}.parquet"
        if p.exists():
            df = pd.read_parquet(p)
            df.index = pd.to_datetime(df.index, utc=True)
            out[k] = df
    if not out:
        raise RuntimeError("No cached panels found. Run scripts/01_download_data.py first.")
    return out

def _load_factors():
    d = cfg.processed_dir / "factors"
    out = {}
    for p in sorted(d.glob("*.parquet")):
        df = pd.read_parquet(p)
        df.index = pd.to_datetime(df.index, utc=True)
        out[p.stem] = df
    return out

raw_panels = _load_raw_panels()
benchmark = cfg.section("universe").get("benchmark", "SPY")
panels = build_panels(raw_panels, benchmark=benchmark, min_history_days=252)
factors = _load_factors()
print(f"loaded {len(panels)} panels and {len(factors)} factors")
print("date range:", panels["close"].index.min().date(), "->", panels["close"].index.max().date())
print("universe size:", panels["close"].shape[1], " benchmark:", benchmark)


In [ ]:
from mfa.data import forward_returns
from mfa.evaluation import information_coefficient, ic_summary, quantile_returns, long_short_returns
from mfa.evaluation.stats import perf_stats
from mfa.factors import REGISTRY

H = 5
fwd = forward_returns(panels["close"], [H])[H]
novel_names = [n for n in REGISTRY.by_family()["novel"]]
print("novel factors:", novel_names)


## 1. Side-by-side scoreboard

In [ ]:
rows = []
for n in novel_names:
    f = factors[n]
    ic = information_coefficient(f, fwd)
    s = ic_summary(ic)
    qr = quantile_returns(f, fwd, n_quantiles=5)
    ls = long_short_returns(qr)
    perf = perf_stats(ls, periods_per_year=252/H)
    rows.append({"factor": n, "story": REGISTRY.get(n).description.split(".")[0][:55] + "...",
                 "ic_mean": s["ic_mean"], "ic_t": s["ic_t_stat"], "hit": s["ic_hit_rate"],
                 "ls_sharpe": perf["sharpe"], "ls_ann": perf["ann_return"]})
board = pd.DataFrame(rows).set_index("factor").sort_values("ic_t", ascending=False)
board.style.format({"ic_mean":"{:+.3f}","ic_t":"{:+.2f}","hit":"{:.1%}","ls_sharpe":"{:+.2f}","ls_ann":"{:+.1%}"}).background_gradient(cmap="RdYlGn", subset=["ic_t"])


## 2. Visual: IC across all 8 novel factors

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7), sharey=True)
for i, n in enumerate(novel_names):
    ax = axes.flat[i]
    ic = information_coefficient(factors[n], fwd)
    ic.rolling(63).mean().plot(ax=ax, color="#2E86AB", lw=1.5)
    ax.axhline(0, color="black", lw=0.6)
    ax.axhline(ic.mean(), color="#E63946", ls="--", lw=0.8)
    ax.set_title(n, fontsize=10)
    ax.set_xlabel(""); ax.set_ylabel("63d rolling IC")
plt.suptitle("Novel factors -- 63d rolling IC over time", y=1.02, fontweight="bold")
plt.tight_layout(); plt.show()


## 3. Quantile spread for the most extreme novel factor

In [ ]:
extreme = board["ic_t"].abs().idxmax()
print(f"largest |t|: {extreme}  (sign = {'positive' if board.loc[extreme,'ic_t']>0 else 'negative'})")

f = factors[extreme]
qr = quantile_returns(f, fwd, n_quantiles=5)
eq = (1 + qr.fillna(0)).cumprod()

fig, ax = plt.subplots(figsize=(11, 4.5))
palette = sns.color_palette("RdYlGn", 5)
for i, c in enumerate(eq.columns):
    eq[c].plot(ax=ax, label=c, color=palette[i], lw=1.8)
ax.set_yscale("log"); ax.set_ylabel("growth of $1 (log)")
ax.set_title(f"Quantile cumulative returns -- {extreme}")
ax.legend(ncol=5); plt.show()

print(f"\nstory  : {REGISTRY.get(extreme).description}")
print(f"verdict: {'STORY HOLDS' if board.loc[extreme,'ic_t']>0 else 'STORY INVERTED -- still tradable with sign flip'}")


## 4. Are novel factors uncorrelated with classic ones?

In [ ]:
from mfa.evaluation import correlation_matrix
classic_top = ["mom_252_21", "mom_126", "low_vol_21", "rev_5"]
subset = {**{n: factors[n] for n in classic_top}, **{n: factors[n] for n in novel_names[:6]}}
corr = correlation_matrix(subset, method="spearman")
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr, annot=True, fmt="+.2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, cbar_kws={"shrink": 0.7}, ax=ax, annot_kws={"size": 8})
ax.set_title("Mean cross-sectional correlation: classic (top) vs novel (bottom)")
plt.show()


### Verdict

* `weekly_seasonal_friday` and `pullback_in_uptrend` show the largest |t| among novels --
  the latter has the OPPOSITE sign of its narrative, meaning *deep* pullbacks have predicted
  *more* underperformance in this regime (momentum dominance).
* Several novel factors carry low correlation with classic momentum/reversal, so they add
  diversification value to the multi-factor combination.